# 01 — Dataset Exploration (MalariaAI, Phase 3)

Systematic exploration of the **real** dataset before any preprocessing design.
Everything printed here is computed from your files; results are saved to
`project_reports/evidence/dataset/` and merged into `project_reports/metrics/dataset_metrics.json`.

**Test-set discipline:** integrity checks (corruption, duplicates, leakage) use all images.
Anything that could *inform a design decision* (pixel statistics, visual samples, shortcut analysis) uses the **train split only**.

**Prerequisites:** the split manifests from `python -m src.data.splitter` must exist in `data/splits/`.
Set the dataset location with `$env:MALARIA_DATA_DIR` or `DATA_DIR_OVERRIDE` below.

In [1]:
import os, io, re, json, hashlib, time, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

# ---- locate the project root (folder containing config.yaml), independent of where Jupyter started ----
ROOT = Path.cwd().resolve()
while not (ROOT / "config.yaml").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("config.yaml not found: start Jupyter inside the project folder.")
    ROOT = ROOT.parent
os.chdir(ROOT)
import sys; sys.path.insert(0, str(ROOT))
from src.config import load_config

DATA_DIR_OVERRIDE = r"D:\CNN-Malaria-Detection-master\data\cell_images"   # e.g. r"D:\path\to\cell_images"  (or set $env:MALARIA_DATA_DIR)
cfg = load_config()
DATA_DIR = Path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE else cfg["abs_paths"]["data_dir"]
SPLITS_DIR = cfg["abs_paths"]["splits_dir"]
EVID = ROOT / "project_reports" / "evidence" / "dataset"
METRICS = ROOT / "project_reports" / "metrics" / "dataset_metrics.json"
EVID.mkdir(parents=True, exist_ok=True)
SEED = cfg["training"]["random_seed"]
CLASSES = cfg["classes"]["names"]; POS = cfg["classes"]["positive_class"]
assert DATA_DIR.is_dir(), f"Dataset folder not found: {DATA_DIR}"
assert (SPLITS_DIR / "manifest.csv").exists(), "Run: python -m src.data.splitter first"
print("project root :", ROOT); print("dataset      :", DATA_DIR); print("splits       :", SPLITS_DIR)

RESULTS = {"generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
           "seed": SEED, "dataset_dir_name": DATA_DIR.name}
def py(o):
    if isinstance(o, dict): return {str(k): py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [py(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, np.ndarray): return py(o.tolist())
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return o

ModuleNotFoundError: No module named 'numpy'

## 1. Dataset discovery and manifest consistency

In [ ]:
manifest = pd.read_csv(SPLITS_DIR / "manifest.csv")
disk = {}
for cls in CLASSES:
    files = [f for f in (DATA_DIR / cls).iterdir() if f.is_file()]
    imgs = [f for f in files if f.suffix.lower() in {".png", ".jpg", ".jpeg"}]
    disk[cls] = {"files_total": len(files), "images": len(imgs),
                 "extensions": pd.Series([f.suffix.lower() for f in files]).value_counts().to_dict()}
print(json.dumps(disk, indent=2))
other_dirs = [d.name for d in DATA_DIR.iterdir() if d.is_dir() and d.name not in CLASSES]
disk_paths = {f"{c}/{f.name}" for c in CLASSES for f in (DATA_DIR / c).iterdir()
              if f.is_file() and f.suffix.lower() in {".png", ".jpg", ".jpeg"}}
man_paths = set(manifest["path"])
print("other folders in dataset dir:", other_dirs)
print("images on disk:", len(disk_paths), "| in manifest:", len(man_paths),
      "| on disk but not in manifest:", len(disk_paths - man_paths),
      "| in manifest but missing on disk:", len(man_paths - disk_paths))
RESULTS["discovery"] = {"per_class": disk, "other_folders": other_dirs, "images_on_disk": len(disk_paths),
    "manifest_rows": len(manifest), "disk_not_in_manifest": len(disk_paths - man_paths),
    "manifest_not_on_disk": len(man_paths - disk_paths)}

## 2. Full scan of every image
One pass reads each file once and records: size, mode, format, readability, MD5, per-image colour statistics,
the fraction of near-black pixels (background), and a 256-bit average hash (near-duplicate candidates).
Pixel sums are kept so dataset-level channel statistics can be computed from the **train split only**.
This takes a few minutes for ~27k images.

In [ ]:
def scan_one(rel):
    rec = {"path": rel}
    try:
        raw = (DATA_DIR / rel).read_bytes()
        rec["md5"] = hashlib.md5(raw).hexdigest()
        im = Image.open(io.BytesIO(raw)); im.load()
        rec.update(width=im.width, height=im.height, mode=im.mode, format=im.format)
        a = np.asarray(im.convert("RGB"), dtype=np.float64)
        n = a.shape[0] * a.shape[1]
        ch_mean = a.reshape(-1, 3).mean(0); ch_std = a.reshape(-1, 3).std(0)
        lum = a @ np.array([0.299, 0.587, 0.114])
        rec.update(mean_r=ch_mean[0], mean_g=ch_mean[1], mean_b=ch_mean[2],
                   std_r=ch_std[0], std_g=ch_std[1], std_b=ch_std[2],
                   brightness=lum.mean(), contrast=lum.std(),
                   min_px=a.min(), max_px=a.max(),
                   black_frac=float((a.max(axis=2) < 10).mean()),
                   chan_diff=float(np.abs(a[..., 0] - a[..., 1]).mean() + np.abs(a[..., 1] - a[..., 2]).mean()),
                   n_pixels=n,
                   s1_r=a[..., 0].sum(), s1_g=a[..., 1].sum(), s1_b=a[..., 2].sum(),
                   s2_r=(a[..., 0] ** 2).sum(), s2_g=(a[..., 1] ** 2).sum(), s2_b=(a[..., 2] ** 2).sum())
        g = np.asarray(im.convert("L").resize((16, 16), Image.BILINEAR), dtype=np.float64)
        rec["ahash"] = "".join("1" if v else "0" for v in (g > g.mean()).ravel())
        rec["readable"] = True
    except Exception as e:
        rec["readable"] = False; rec["error"] = type(e).__name__
    return rec

t0 = time.time()
paths = manifest["path"].tolist()
with ThreadPoolExecutor(max_workers=min(8, os.cpu_count() or 4)) as ex:
    recs = []
    for i, r in enumerate(ex.map(scan_one, paths, chunksize=64), 1):
        recs.append(r)
        if i % 5000 == 0: print(f"  scanned {i}/{len(paths)}")
scan = pd.DataFrame(recs).merge(manifest, on="path", how="left")
print(f"done in {time.time() - t0:.0f}s | rows: {len(scan)}")
scan["aspect"] = scan["width"] / scan["height"]
scan["area"] = scan["width"] * scan["height"]
keep = [c for c in scan.columns if not c.startswith(("s1_", "s2_"))]
scan[keep].round(3).to_csv(EVID / "image_scan.csv", index=False)
print("saved", EVID / "image_scan.csv")

## 3. Corrupted / unreadable images, formats and colour modes

In [ ]:
bad = scan[~scan["readable"]]
ok = scan[scan["readable"]].copy()
print("unreadable:", len(bad)); print(bad[["path", "error"]].head(10).to_string(index=False) if len(bad) else "")
print("formats:", ok["format"].value_counts().to_dict()); print("modes  :", ok["mode"].value_counts().to_dict())
gray_like = ok[ok["chan_diff"] < 1.0]
print("grayscale-like images stored as RGB (channel difference < 1.0):", len(gray_like))
RESULTS["integrity"] = {"unreadable": len(bad), "unreadable_examples": bad["path"].head(10).tolist(),
    "formats": ok["format"].value_counts().to_dict(), "modes": ok["mode"].value_counts().to_dict(),
    "grayscale_like_images": len(gray_like),
    "pixel_value_range": [float(ok["min_px"].min()), float(ok["max_px"].max())]}
print("pixel value range:", RESULTS["integrity"]["pixel_value_range"])

## 4. Image dimensions and aspect ratios

In [ ]:
def desc(s): return {"min": float(s.min()), "p5": float(s.quantile(.05)), "median": float(s.median()),
                     "mean": float(s.mean()), "p95": float(s.quantile(.95)), "max": float(s.max())}
dims = {"width": desc(ok["width"]), "height": desc(ok["height"]), "aspect_w_over_h": desc(ok["aspect"])}
by_class = {c: {"width_median": float(g["width"].median()), "height_median": float(g["height"].median()),
                "aspect_median": float(g["aspect"].median())} for c, g in ok.groupby("class")}
non_square = float(((ok["aspect"] - 1).abs() > 0.10).mean())
extreme = ok[(ok["aspect"] < 0.6) | (ok["aspect"] > 1.67)]
small = ok[(ok["width"] < 64) | (ok["height"] < 64)]
print(pd.DataFrame(dims).T.round(2).to_string()); print(pd.DataFrame(by_class).T.round(2).to_string())
print(f"unique (w,h) sizes: {ok[['width','height']].drop_duplicates().shape[0]}")
print(f"images with |aspect-1| > 0.10: {non_square:.1%} | extreme aspect (<0.6 or >1.67): {len(extreme)} | smaller than 64 px on a side: {len(small)}")
print(f"images smaller than 224 px on both sides (would be upscaled): {((ok['width']<224)&(ok['height']<224)).mean():.1%}")
RESULTS["dimensions"] = {**dims, "by_class": by_class, "unique_sizes": int(ok[['width','height']].drop_duplicates().shape[0]),
    "fraction_non_square_gt10pct": non_square, "extreme_aspect_count": len(extreme), "smaller_than_64px_count": len(small),
    "fraction_smaller_than_224_both_sides": float(((ok['width']<224)&(ok['height']<224)).mean())}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for c, g in ok.groupby("class"):
    ax[0].hist(g["width"], bins=50, alpha=.55, label=c); ax[1].hist(g["height"], bins=50, alpha=.55, label=c)
    ax[2].hist(g["aspect"], bins=50, alpha=.55, label=c)
for a_, t in zip(ax, ["Width (px)", "Height (px)", "Aspect ratio (w/h)"]): a_.set_title(t); a_.legend()
plt.tight_layout(); plt.savefig(EVID / "image_size_and_aspect.png", dpi=130); plt.show()

## 5. Duplicate analysis

In [ ]:
dup_groups = ok.groupby("md5")["path"].apply(list); dup_groups = dup_groups[dup_groups.str.len() > 1]
md5_cross = ok[ok.duplicated("md5", keep=False)].groupby("md5")["split"].nunique(); md5_cross = md5_cross[md5_cross > 1]
ah = ok.groupby(["ahash", "width", "height"]).agg(n=("path", "size"), splits=("split", "nunique"), classes=("label", "nunique"))
ah_multi = ah[ah["n"] > 1]
print(f"exact duplicates (MD5): {len(dup_groups)} groups | spanning more than one split: {len(md5_cross)}")
print(f"NEAR-DUPLICATE CANDIDATES (same 16x16 average hash AND identical width/height; not confirmed): {len(ah_multi)} groups covering "
      f"{int(ah_multi['n'].sum()) if len(ah_multi) else 0} images | across splits: {int((ah_multi['splits']>1).sum())} "
      f"| across both classes: {int((ah_multi['classes']>1).sum())}")
RESULTS["duplicates"] = {"exact_md5_groups": len(dup_groups), "exact_md5_groups_crossing_splits": len(md5_cross),
    "exact_examples": dup_groups.head(5).tolist(),
    "ahash_candidate_groups": len(ah_multi), "ahash_candidate_images": int(ah_multi["n"].sum()) if len(ah_multi) else 0,
    "ahash_candidates_crossing_splits": int((ah_multi["splits"] > 1).sum()),
    "ahash_candidates_crossing_classes": int((ah_multi["classes"] > 1).sum()),
    "note": "Candidates = identical 16x16 average hash and identical image size. They need visual review; they are not proof of duplication."}

## 6. Class distribution by split (and patients)

In [ ]:
tab = ok.pivot_table(index="split", columns="class", values="path", aggfunc="count").reindex(["train", "val", "test"])
tab["total"] = tab.sum(axis=1); tab["parasitized_fraction"] = (tab.get(POS, 0) / tab["total"]).round(4)
tab["patients"] = ok.groupby("split")["patient_id"].nunique(); print(tab.to_string())
per_patient = ok.groupby("patient_id").agg(n=("path", "size"), parasitized=("label", "sum"))
per_patient["uninfected"] = per_patient["n"] - per_patient["parasitized"]
both = int(((per_patient["parasitized"] > 0) & (per_patient["uninfected"] > 0)).sum())
only_p = int(((per_patient["parasitized"] > 0) & (per_patient["uninfected"] == 0)).sum())
only_u = int(((per_patient["parasitized"] == 0) & (per_patient["uninfected"] > 0)).sum())
print(f"patients: {len(per_patient)} | with both classes: {both} | only parasitized: {only_p} | only uninfected: {only_u}")
print("images per patient:", {k: round(v, 1) for k, v in desc(per_patient["n"]).items()})
RESULTS["class_distribution"] = {"table": tab.reset_index().to_dict(orient="records"),
    "patients_total": len(per_patient), "patients_with_both_classes": both, "patients_only_parasitized": only_p,
    "patients_only_uninfected": only_u, "images_per_patient": desc(per_patient["n"])}
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
tab[[c for c in CLASSES if c in tab.columns]].plot(kind="bar", stacked=True, ax=ax[0], rot=0); ax[0].set_title("Images per split and class")
ax[1].hist(per_patient["n"], bins=40); ax[1].set_title("Images per patient group"); ax[1].set_xlabel("images")
plt.tight_layout(); plt.savefig(EVID / "class_distribution_by_split.png", dpi=130); plt.show()

## 7. Pixel statistics and colour consistency
Dataset-level channel mean/std come from the **train split only**. (MobileNetV2 uses its own scaling, so these are
for the custom CNN and for documentation.) Per-patient colour variation is a proxy for staining/illumination differences.

In [ ]:
tr = ok[ok["split"] == "train"]
n = tr["n_pixels"].sum()
mean = np.array([tr["s1_r"].sum(), tr["s1_g"].sum(), tr["s1_b"].sum()]) / n
std = np.sqrt(np.array([tr["s2_r"].sum(), tr["s2_g"].sum(), tr["s2_b"].sum()]) / n - mean ** 2)
print("train-set channel mean (0-255):", mean.round(2), "| std:", std.round(2))
print("train-set channel mean (0-1)  :", (mean / 255).round(4), "| std:", (std / 255).round(4))
cls_stats = ok[ok["split"] == "train"].groupby("class")[["mean_r", "mean_g", "mean_b", "brightness", "contrast", "black_frac"]].mean().round(2)
print(cls_stats.to_string())
pat_b = ok.groupby("patient_id")["brightness"].mean()
print("patient-level mean brightness: min %.1f | median %.1f | max %.1f | std %.1f" % (pat_b.min(), pat_b.median(), pat_b.max(), pat_b.std()))
dark = ok[ok["brightness"] < ok["brightness"].quantile(.01)]; bright = ok[ok["brightness"] > ok["brightness"].quantile(.99)]
print("black-background fraction: median %.3f | p95 %.3f | images with >50%% near-black pixels: %d" %
      (ok["black_frac"].median(), ok["black_frac"].quantile(.95), int((ok["black_frac"] > .5).sum())))
RESULTS["pixel_statistics"] = {"computed_on": "train split only", "channel_mean_0_255": mean, "channel_std_0_255": std,
    "channel_mean_0_1": mean / 255, "channel_std_0_1": std / 255, "per_class_train_means": cls_stats.reset_index().to_dict(orient="records"),
    "patient_brightness": {"min": pat_b.min(), "median": pat_b.median(), "max": pat_b.max(), "std": pat_b.std()},
    "black_fraction_median": ok["black_frac"].median(), "images_over_50pct_black": int((ok["black_frac"] > .5).sum())}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for c, g in tr.groupby("class"):
    ax[0].hist(g["brightness"], bins=50, alpha=.55, label=c); ax[1].hist(g["black_frac"], bins=50, alpha=.55, label=c)
ax[0].set_title("Mean brightness per image (train)"); ax[1].set_title("Near-black pixel fraction (train)"); ax[0].legend()
ax[2].hist(pat_b, bins=40); ax[2].set_title("Mean brightness per patient (all splits)")
plt.tight_layout(); plt.savefig(EVID / "pixel_statistics.png", dpi=130); plt.show()

## 8. Shortcut check: can simple image properties alone predict the label?
For each property the table shows the AUC for separating Parasitized from Uninfected on the **train split**
(0.5 = no information). Size/shape properties (width, height, aspect, area) have no biological reason to carry the label,
so values far from 0.5 would warn of an acquisition artefact a network could exploit. Colour/brightness differences can be
partly genuine (stain) so they are reported, not judged.

In [ ]:
def rank_auc(x, y):
    r = pd.Series(x).rank(method="average").to_numpy(); y = np.asarray(y)
    n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
feats = ["width", "height", "aspect", "area", "brightness", "contrast", "mean_r", "mean_g", "mean_b", "black_frac"]
rows = [{"feature": f, "auc_parasitized_higher": round(rank_auc(tr[f].values, tr["label"].values), 4)} for f in feats]
shortcut = pd.DataFrame(rows); shortcut["distance_from_0.5"] = (shortcut["auc_parasitized_higher"] - .5).abs().round(4)
shortcut = shortcut.sort_values("distance_from_0.5", ascending=False)
print(shortcut.to_string(index=False)); shortcut.to_csv(EVID / "shortcut_features_auc.csv", index=False)
RESULTS["shortcut_check"] = {"computed_on": "train split only", "table": shortcut.to_dict(orient="records")}

## 9. Representative samples (train split, fixed seed)

In [ ]:
rng = np.random.RandomState(SEED)
for cls in CLASSES:
    pool = tr[tr["class"] == cls]["path"].tolist(); pick = rng.choice(pool, size=min(32, len(pool)), replace=False)
    fig, axes = plt.subplots(4, 8, figsize=(16, 8.4))
    for a_, p in zip(axes.ravel(), pick):
        a_.imshow(Image.open(DATA_DIR / p).convert("RGB")); a_.set_title(Path(p).stem[:14], fontsize=6); a_.axis("off")
    for a_ in axes.ravel()[len(pick):]: a_.axis("off")
    fig.suptitle(f"{cls} — random train samples (native aspect ratio)"); plt.tight_layout()
    plt.savefig(EVID / f"samples_{cls.lower()}.png", dpi=110); plt.show()

## 10. Independent verification that patient groups do not cross splits
This re-derives patient IDs **without** the splitter's regex (everything before the first `thin`, case-insensitive),
and re-reads the three split CSVs separately from `manifest.csv`. It also reports the *stricter* smear-level overlap and an
informational check on whether IDs that differ only by a trailing letter (e.g. `...P141` vs `...P141N`) sit in different splits.

In [ ]:
splits = {s: pd.read_csv(SPLITS_DIR / f"{s}.csv") for s in ["train", "val", "test"]}
def indep_id(path):
    name = Path(path).stem
    return re.split("thin", name, maxsplit=1, flags=re.IGNORECASE)[0].rstrip("_").upper()
for s, d in splits.items(): d["indep_id"] = d["path"].map(indep_id)
checks = {}
sets = {s: set(d["indep_id"]) for s, d in splits.items()}
checks["independent_patient_id_overlap_train_val"] = len(sets["train"] & sets["val"])
checks["independent_patient_id_overlap_train_test"] = len(sets["train"] & sets["test"])
checks["independent_patient_id_overlap_val_test"] = len(sets["val"] & sets["test"])
sm = {s: set(d["smear_id"]) for s, d in splits.items()}
checks["smear_overlap_train_val"] = len(sm["train"] & sm["val"]); checks["smear_overlap_train_test"] = len(sm["train"] & sm["test"])
checks["smear_overlap_val_test"] = len(sm["val"] & sm["test"])
pp = {s: set(d["path"]) for s, d in splits.items()}
checks["path_overlap_any"] = len(pp["train"] & pp["val"]) + len(pp["train"] & pp["test"]) + len(pp["val"] & pp["test"])
union = pp["train"] | pp["val"] | pp["test"]
checks["images_in_splits"] = sum(len(v) for v in pp.values()); checks["unique_images_in_splits"] = len(union)
checks["splits_cover_all_disk_images"] = bool(union == disk_paths)
checks["manifest_split_column_matches_split_files"] = bool(
    all(set(manifest[manifest["split"] == s]["path"]) == pp[s] for s in pp))
base = lambda i: re.sub(r"[A-Z]+$", "", i)
bs = {s: {base(i) for i in v} for s, v in sets.items()}
checks["INFO_base_id_overlap_(letter-suffix variants)_train_val"] = len(bs["train"] & bs["val"])
checks["INFO_base_id_overlap_train_test"] = len(bs["train"] & bs["test"]); checks["INFO_base_id_overlap_val_test"] = len(bs["val"] & bs["test"])
for k, v in checks.items(): print(f"{k:70s} {v}")
hard = {k: v for k, v in checks.items() if "overlap" in k and not k.startswith("INFO")}
passed = all(v == 0 for v in hard.values()) and checks["splits_cover_all_disk_images"] and checks["manifest_split_column_matches_split_files"]
print("\nLEAKAGE VERIFICATION:", "PASS" if passed else "FAIL")
sumfile = SPLITS_DIR / "split_summary.json"
if sumfile.exists():
    ss = json.loads(sumfile.read_text()); recon = all(ss["splits"][s]["images"] == len(splits[s]) for s in splits)
    print("split_summary.json image counts match the CSVs:", recon); checks["split_summary_matches_csvs"] = recon
    RESULTS["split_summary_from_splitter"] = ss
RESULTS["leakage_verification"] = {"passed": passed, "checks": checks,
    "interpretation_of_INFO": "Non-zero INFO overlap means IDs differing only by a trailing letter appear in different splits. This is only a leak if those IDs are the same person, which the filenames alone cannot establish."}

## 11. Save results

In [ ]:
(EVID / "exploration_summary.json").write_text(json.dumps(py(RESULTS), indent=2), encoding="utf-8")
metrics = json.loads(METRICS.read_text(encoding="utf-8")) if METRICS.exists() else {}
metrics["exploration"] = py(RESULTS); metrics["exploration_source"] = "notebooks/01_dataset_exploration.ipynb (run by the user)"
METRICS.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("saved:", *sorted(p.name for p in EVID.iterdir()), sep="\n  ")
print("updated:", METRICS)

## 12. Findings (fill in after reading the outputs above)
Record here, in your own words, anything that changes preprocessing: image-size handling (pad vs stretch, upscaling),
black-background proportion, size/colour shortcuts, patient-level colour variation, duplicate candidates.
These feed directly into Phase 4 (`src/data/preprocessing.py`).